In [22]:
#!pip install tabulate 

In [23]:
import pandas as pd

In [24]:
# leitura 
arquivo = "/home/fabiene/Documentos/Fabiene/UDEMY_ML/LSTM_AI/Modulo6/NYW.csv"
df_leitura = pd.read_csv(arquivo, header=None, names=['codigo', 'local','data', 'vento_em_nos'])
df_leitura.head()


,codigo,local,data,vento_em_nos
0,USW00014732,"LA GUARDIA AIRPORT, NY US",7/1/2017,9.84
1,USW00014732,"LA GUARDIA AIRPORT, NY US",7/2/2017,8.72
2,USW00014732,"LA GUARDIA AIRPORT, NY US",7/3/2017,7.83
3,USW00014732,"LA GUARDIA AIRPORT, NY US",7/4/2017,4.92
4,USW00014732,"LA GUARDIA AIRPORT, NY US",7/5/2017,9.62


In [25]:
df_leitura['data'] = pd.to_datetime(df_leitura['data'])

In [26]:
df_leitura['codigo'].unique()

array(['USW00014732', 'USW00014734'], dtype=object)

In [27]:
df_leitura['local'].unique()

array(['LA GUARDIA AIRPORT, NY US',
       'NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US'], dtype=object)

In [28]:
df_leitura.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   codigo        732 non-null    object        
 1   local         732 non-null    object        
 2   data          732 non-null    datetime64[ns]
 3   vento_em_nos  732 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 23.0+ KB


In [29]:
from datetime import datetime, timedelta, date


def alerta_menor_variacao_media_movel_7dias(df, n_weeks = 4, variation=0.2):
    '''
    queda maior de x% na média do mesmo dia nas últimas 4 semanas. Alertar apenas últimos 7 dias.
    n_weeks = 4

    Parâmetros:
    - variation: variação em percentual máxima

    Retorna:
    alerts: lista com as datas da série onde houve variação.
    '''
    df = df.copy()
    df.columns = df.columns = ['dt','x']

    # Calcula a média móvel dos valores deslocados por semanas passadas
    #df['expected_x'] = df['x'].rolling(window=n_weeks*7, min_periods=n_weeks*7, step=7).mean()
    # média móvel de 7 dias, em um dia ->
    df['expected_x'] = df['x'].rolling(window=7, min_periods=1).mean()


   # print(df['expected_x'])
   # print((df['x']/df['expected_x'])-1)
    #  Calcula a média dos valores correspondentes de 'x' nas últimas n_weeks semanas para cada data.
    #for w in range(1,(n_weeks+1)): Cria um loop que itera pelas semanas passadas (1 semana atrás, 2 semanas atrás, ..., n_weeks semanas atrás).
    #df['x'].shift(w*7): Desloca a coluna 'x' para trás em w * 7 dias, ou seja, pega os valores de 'x' das semanas anteriores. Por exemplo, se w for 1, ele pega os valores de 'x' de 7 dias atrás; se w for 2, pega os valores de 14 dias atrás.
    #sum([...]): Soma os valores deslocados de 'x' de cada semana anterior.
    #/n_weeks: Divide a soma pela quantidade de semanas (n_weeks) para calcular a média, que n caso é para calcular a média móvel no final.
    #O resultado dessa média é atribuído à nova coluna 'expected_x' do df.

    df['var'] = (df['x']/df['expected_x'])-1
    # Calcula a variação percentual entre o valor atual ('x') e a média esperada ('expected_x').
    # df['x']/df['expected_x']: Calcula a razão entre o valor atual e a média esperada.
    #-1: Subtrai 1 do resultado para obter a variação percentual (ex: 0.1 = 10%, -0.2 = -20%).

    df['alarm'] = df['var']> variation # acima de 20% de variação
    #df['alarm'] = df['var']<-variation
    #print('final')
    #print(df)
    return (
        df
        .tail(20) # últimos 20 dias focados.
        .loc[
            df.tail(20)['alarm'],  # últimos 20 dias focados.
            'dt'
        ].tolist()
    )


# ----------------------------------------------------------------------------------------------------------------------------------


# investiga alertas anteriores
def Varrer_sensor_menor_teste(df_ga_investigar, variavel_porcento, Valor_pesquisa, lista_interesse):
    # Coluna para agrupar
    #cols_to_group = ['local']
   
    cols_to_group = lista_interesse
    #cols_to_group = ['local', 'codigo']
    # Dicionário de alarmes
    alarms = {
        alerta_menor_variacao_media_movel_7dias: {'variation': variavel_porcento}
    }
   
    # Agrupando o DataFrame pela coluna especificada
    dfs_to_calc = df_ga_investigar.groupby(by=cols_to_group)
   
    eval_alertas_menor = []
   
    for group, df_calc in dfs_to_calc:
        for alarm, alarm_params in alarms.items():
            result_alarm = alarm(df_calc[['data', f'{Valor_pesquisa}']], **alarm_params)
           
            for dt_alarm in result_alarm:
                value_at_alarm = df_calc[df_calc['data'] == dt_alarm][f'{Valor_pesquisa}'].iloc[0]
                tmp = {
                    **dict(zip(cols_to_group, [group])), # quando é apenas uma palavra na lista =lista_interesse
                   # **dict(zip(cols_to_group, group)), # correcao, quando é mais de 2
                    'alarme': alarm.__name__,
                    'data': dt_alarm,
                    f'{Valor_pesquisa}': value_at_alarm
                }
                eval_alertas_menor.append(tmp)
   
    df_alertas_menor = pd.DataFrame(eval_alertas_menor)
    if not df_alertas_menor.empty:
        #display(df_alertas_menor)
        df_alertas_menor = df_alertas_menor.loc[
            df_alertas_menor['data'] >= (df_alertas_menor['data'].max() - timedelta(days=6))
        ]
    else :
      df_alertas_menor=pd.DataFrame()

    return df_alertas_menor

In [30]:

def alerta_ATUAL_menor_variacao_mediamovel7dias(df, n_weeks = 4, variation=0.2):
    '''
    queda maior de x% na média do mesmo dia nas últimas 4 semanas. Alertar apenas últimos 7 dias.
    n_weeks = 4

    Parâmetros:
    - variation: variação em percentual máxima

    Retorna:
    alerts: lista com as datas da série onde houve variação.
    '''
    df = df.copy()
    print(df.columns)
    print(df)
    print(df.info())
    df.columns = df.columns = ['dt','x']
    # validação atual
    df['expected_x'] = sum([
        df['x'].shift(w*7) # Pega o valor de 'x' de w semanas atrás (no mesmo dia da semana)
        for w in range(1,(n_weeks+1)) # Itera de w=1 até w=n_weeks
    ])/n_weeks # Divide a soma pelo número de semanas para obter a média

    #Exemplo prático: Imagine que hoje é uma Quarta-feira, e n_weeks = 4.
    #df['x'] para hoje (Quarta-feira atual) é o valor que estamos analisando.
    #df['expected_x'] para hoje será a média dos valores de x das últimas 4 Quartas-feiras.


    # Calcula a média móvel dos valores deslocados por semanas passadas
    #df['expected_x'] = df['x'].rolling(window=n_weeks*7, min_periods=n_weeks*7, step=7).mean()
    # quando há atraso nos dados abaixo !!!!!
    # média móvel de 7 dias, em um dia -> anterior feito antes
    #df['expected_x'] = df['x'].rolling(window=7, min_periods=1).mean() # depois volto ao anterior


   # print(df['expected_x'])
   # print((df['x']/df['expected_x'])-1)
    #  Calcula a média dos valores correspondentes de 'x' nas últimas n_weeks semanas para cada data.
    #for w in range(1,(n_weeks+1)): Cria um loop que itera pelas semanas passadas (1 semana atrás, 2 semanas atrás, ..., n_weeks semanas atrás).
    #df['x'].shift(w*7): Desloca a coluna 'x' para trás em w * 7 dias, ou seja, pega os valores de 'x' das semanas anteriores. Por exemplo, se w for 1, ele pega os valores de 'x' de 7 dias atrás; se w for 2, pega os valores de 14 dias atrás.
    #sum([...]): Soma os valores deslocados de 'x' de cada semana anterior.
    #/n_weeks: Divide a soma pela quantidade de semanas (n_weeks) para calcular a média, que n caso é para calcular a média móvel no final.
    #O resultado dessa média é atribuído à nova coluna 'expected_x' do df.

    df['var'] = (df['x']/df['expected_x'])-1
    # Calcula a variação percentual entre o valor atual ('x') e a média esperada ('expected_x').
    # df['x']/df['expected_x']: Calcula a razão entre o valor atual e a média esperada.
    #-1: Subtrai 1 do resultado para obter a variação percentual (ex: 0.1 = 10%, -0.2 = -20%).

    df['alarm'] = df['var']<-variation
    print('final')
    print(df)
    return (
        df
        .tail(7) # últimos 7 dias focados.
        .loc[
            df.tail(7)['alarm'],  # últimos 7 dias focados.
            'dt'
        ].tolist()
    )


# ----------------------------------------------------------------------------



# investiga alertas anteriores
def Varrer_sensor_menor(df_ga_investigar, variavel_porcento, Valor_pesquisa, lista_interesse):
    # Coluna para agrupar
    #cols_to_group = ['local']
   
    cols_to_group = lista_interesse
    #cols_to_group = ['local', 'codigo']
    # Dicionário de alarmes
    alarms = {
        alerta_ATUAL_menor_variacao_mediamovel7dias: {'variation': variavel_porcento}
    }
   
    # Agrupando o DataFrame pela coluna especificada
    dfs_to_calc = df_ga_investigar.groupby(by=cols_to_group)
   
    eval_alertas_menor = []
   
    for group, df_calc in dfs_to_calc:
        for alarm, alarm_params in alarms.items():
            result_alarm = alarm(df_calc[['data', f'{Valor_pesquisa}']], **alarm_params)
           
            for dt_alarm in result_alarm:
                value_at_alarm = df_calc[df_calc['data'] == dt_alarm][f'{Valor_pesquisa}'].iloc[0]
                tmp = {
                    **dict(zip(cols_to_group, [group])), # quando é apenas uma palavra na lista =lista_interesse
                   # **dict(zip(cols_to_group, group)), # correcao, quando é mais de 2
                    'alarme': alarm.__name__,
                    'data': dt_alarm,
                    f'{Valor_pesquisa}': value_at_alarm
                }
                eval_alertas_menor.append(tmp)
   
    df_alertas_menor = pd.DataFrame(eval_alertas_menor)
    if not df_alertas_menor.empty:
        display(df_alertas_menor)
        df_alertas_menor = df_alertas_menor.loc[
            df_alertas_menor['data'] >= (df_alertas_menor['data'].max() - timedelta(days=6))
        ]
    else :
      df_alertas_menor=pd.DataFrame()

    return df_alertas_menor


In [31]:
# Principal estatistico

# 1.0) considerado 20% para a variação

lista_interesse_a= ['local']

# Exemplo de uso troquei 0.1 para 0.2 (20%)
df_alertas_20_menor_vento = Varrer_sensor_menor(df_leitura, 0.02, 'vento_em_nos', lista_interesse_a)
print('-'*30)


Index(['data', 'vento_em_nos'], dtype='object')
          data  vento_em_nos
0   2017-07-01          9.84
1   2017-07-02          8.72
2   2017-07-03          7.83
3   2017-07-04          4.92
4   2017-07-05          9.62
..         ...           ...
361 2018-06-27         12.53
362 2018-06-28          9.40
363 2018-06-29          9.17
364 2018-06-30          6.04
365 2018-07-01          5.82

[366 rows x 2 columns]
<class 'pandas.core.frame.DataFrame'>
Index: 366 entries, 0 to 365
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data          366 non-null    datetime64[ns]
 1   vento_em_nos  366 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 8.6 KB
None
final
            dt      x  expected_x       var  alarm
0   2017-07-01   9.84         NaN       NaN  False
1   2017-07-02   8.72         NaN       NaN  False
2   2017-07-03   7.83         NaN       NaN  False
3   201

,local,alarme,data,vento_em_nos
0,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-28,9.40
1,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-30,6.04
2,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01,5.82
3,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-28,8.28
4,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-30,6.93
5,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01,6.04


------------------------------


In [32]:
def agrupar_resultado_para_mostrar(df_alertas_MENOR):
    df_result_email = (
        df_alertas_MENOR
        .groupby(
            [        
                'local',                
               # 'query',
                'alarme',
            ],
            as_index=False
        )
        .agg({
            'data':'max',     #para coletar o último dia para o alerta
        })
    )
    return df_result_email

if not df_alertas_20_menor_vento.empty:
    df_result_ALERTAS_vento = agrupar_resultado_para_mostrar(df_alertas_20_menor_vento)
    display(df_result_ALERTAS_vento)

,local,alarme,data
0,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01
1,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01


In [33]:
df_alertas_20_menor_vento

,local,alarme,data,vento_em_nos
0,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-28,9.40
1,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-30,6.04
2,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01,5.82
3,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-28,8.28
4,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-06-30,6.93
5,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01,6.04


In [34]:
df_result_ALERTAS_vento

,local,alarme,data
0,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01
1,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01


In [35]:
try:
  df_result_ALERTAS_vento = df_result_ALERTAS_vento.merge(df_alertas_20_menor_vento, how='inner', on=[ 'local', 'alarme','data'])
  display(df_result_ALERTAS_vento.tail())

except:
  df_result_ALERTAS_vento = pd.DataFrame()




,local,alarme,data,vento_em_nos
0,"(LA GUARDIA AIRPORT, NY US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01,5.82
1,"(NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,)",alerta_ATUAL_menor_variacao_mediamovel7dias,2018-07-01,6.04


In [36]:
df_leitura.tail(20)

,codigo,local,data,vento_em_nos
712,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-12,7.61
713,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-13,9.84
714,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-14,14.99
715,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-15,10.96
716,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-16,7.61
717,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-17,6.26
718,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-18,12.08
719,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-19,11.63
720,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-20,6.49
721,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2018-06-21,8.28


In [40]:
df_leitura.loc[df_leitura['local'].str.contains('LA GUARDIA AIRPORT, NY US')].tail()

,codigo,local,data,vento_em_nos
361,USW00014732,"LA GUARDIA AIRPORT, NY US",2018-06-27,12.53
362,USW00014732,"LA GUARDIA AIRPORT, NY US",2018-06-28,9.40
363,USW00014732,"LA GUARDIA AIRPORT, NY US",2018-06-29,9.17
364,USW00014732,"LA GUARDIA AIRPORT, NY US",2018-06-30,6.04
365,USW00014732,"LA GUARDIA AIRPORT, NY US",2018-07-01,5.82
